<a href="https://colab.research.google.com/github/rubusarbaro/supplychain-forecast-FIME/blob/main/PIA_Prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

##################################################
##                                              ##
##             PRODUCTO INTEGRADOR              ##
##    Pronósticos en la Cadena de Suministro    ##
##                                              ##
##                                              ##
##  Saúl Roberto Morales Velázquez              ##
##  1856691                                     ##
##                                              ##
##################################################

#**Librerías**

In [ ]:
from bs4 import BeautifulSoup # Permite trabajar con archivos HTML y hacer web scrapping.
from google.colab import userdata # Se utilizará para obtener el token del API de Banxico mediante un secreto.
import json # Permite trabajar con datos en formato JSON.
import numpy as np # Permite trabajar con opercaciones matemáticas avanzadas.
import pandas as pd # Permite trabajar con data frames.
import plotly.express as px # Librería que permite la creación de gráficas interactivas.
import requests # Permite realizar peticiones HTML.

#**Funciones**

In [ ]:
def real_plt(df: object, time_column_name: str, value_column_name: str, title: str, xaxis_title="Tiempo", yaxis_title="Ventas") :
  """
  Grafica la serie de tiempo con los datos reales.

  Args:
      df (object): DataFrame que contiene los datos a graficar.
      time_column_name (str): Nombre de la columna que contiene las fechas.
      value_column_name (str): Nombre de la columna que contiene los valores.
      title (str): Título de la gráfica.

  Returns:
      Object: Gráfica de la serie de tiempo.
  """

  fig = px.line(df, x=time_column_name, y=value_column_name, title=title)

  fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
      buttons=list([
        dict(count=1, label="1m", step="month", stepmode="backward"),
        dict(count=6, label="6m", step="month", stepmode="backward"),
        dict(count=1, label="YTD", step="year", stepmode="todate"),
        dict(count=1, label="1y", step="year", stepmode="backward"),
        dict(step="all")
      ])
    )
  )

  fig.update_layout(
    title = title,
    xaxis_title = "Tiempo",
    yaxis_title = "Tipo de cambio"
  )

  return fig.show()

In [ ]:
def difference_df(df: object, data_column_name: str) :
  """
  Permite obtener la diferencia entre cada fila de datos en elo data frame.

  Args:
      df (object): DataFrame.
      data_column_name (str): Nombre de la columna que contiene los valores.

  Returns:
      Object: DataFrame con la diferencia entre cada fila.
  """

  df_diff = df
  df_diff["diff"] = np.nan

  past_number = 0
  for index, row in df.iterrows():
    if index == 0:
      df_diff.at[index, "diff"] = 0
      past_number = row[data_column_name]
    else:
      df_diff.at[index, "diff"] = (float(row[data_column_name]) / float(past_number))-1
      past_number = row[data_column_name]

  return df

In [ ]:
def get_spike_dates(df: object, date_column_name: str, difference_column_name="diff") :
  avg_diff = np.mean(abs(df[difference_column_name]))
  std_diff = np.std(abs(df[difference_column_name]))

  dates = []
  for index, row in df.iterrows():
    if abs(row[difference_column_name]) > avg_diff + 3*std_diff:
      dates.append(row[date_column_name])

  return dates

In [ ]:
# Función basada en el código de https://scrapfly.io/blog/guide-to-yahoo-finance-api/

def get_YStocks(ticker: str) :
  pass

#**Clases**

In [ ]:
class banxico(API: str) :
  def __init__(self, API: str) :
    self.API = API

#**Códigos HTML**
Permite obtener el estatus del *query*, con el fin de saber si ocurrió algún error al momento de obtener los datos.

In [ ]:
HTTP_codes = {
    "200" : "OK",
    "400" : "Bad Request",
    "401" : "Unauthorized",
    "403" : "Forbidden",
    "404" : "Not Found",
    "500" : "Internal Server Error",
}

#**Obtención de datos mediante API**
Los datos se obtienen de Banxico mediante RestAPI.
Propongo las siguiente series:

*   SF43718 : Tasa de cambio USD/MXN
*   SF60633 : Valor de CETES a 28 días



In [ ]:
serie_ID = "SF60633"  # ID que identifica la serie de tiempo de la tasa de cambio USD/MX FIX.
SIE_API = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{serie_ID}/datos?token={userdata.get('Banxico_Token')}" #API_URL
response = requests.get(SIE_API)  # Usa el método GET para obtener los datos.

HTTP_codes[str(response.status_code)] # Muestra el significado del código de estatus de la petición HTML.

'OK'

#**Data frame**

In [ ]:
data = response.json()  # Almacena los resultados de la petición en una variable, en formato JSON.
series_data = data["bmx"]["series"][0]["datos"] # Escoje únicamente los datos correspondientes a la serie de tiempo, ya que el JSON trae metadatos.
df = pd.DataFrame(series_data)  # Transforma los datos JSON en un data frame.
df.head() # Muetsra solo las primeras líneas del data frame.

,fecha,dato
0,19/09/2006,7.0500
1,26/09/2006,7.0500
2,03/10/2006,7.0500
3,10/10/2006,7.0600
4,17/10/2006,7.0500


In [ ]:
diff_df = difference_df(df, "dato")
spike_dates = get_spike_dates(diff_df, "fecha")
spike_dates

['18/11/2008',
 '24/03/2009',
 '04/01/2011',
 '10/06/2014',
 '06/01/2015',
 '13/01/2015',
 '23/02/2016',
 '05/07/2016',
 '04/10/2016',
 '15/11/2016',
 '23/12/2019',
 '24/03/2020',
 '29/06/2021',
 '01/02/2022',
 '27/09/2022']

In [ ]:
real_plt(df, "fecha", "dato", "Tasa de cambio USD/MXN", yaxis_title="Tipo de cambio")